In [12]:
EMPTY=" "
PLAYER_X="X"
PLAYER_O="O"

def make_board():
    return[" "," "," "," "," "," "," "," "," "]

def print_board(board):
    print()
    print(f" {board[0]} | {board[1]} | {board[2]} ")
    print("---+---+---")
    print(f" {board[3]} | {board[4]} | {board[5]} ")
    print("---+---+---")
    print(f" {board[6]} | {board[7]} | {board[8]} ")
    print()
 


In [13]:
winning_combos =[
    (0,1,2),
    (3,4,5),
    (6,7,8),
    (0,3,6),
    (1,4,7),
    (2,5,8),
    (0,4,8),
    (2,4,6),
]


def check_winner(board):
    for a,b,c in winning_combos:
        if board[a] !=EMPTY and board[a]== board[b]==board[c]:
            return[a]
    return None

def is_board_full(board):
    return EMPTY not in board

def get_empty_cells(board):
    return [i for i in range(9) if board[i]==EMPTY]
    

def opponent(player):
    if player==PLAYER_X:
        return PLAYER_O
    else:
        return PLAYER_X
        


In [15]:
test = make_board()
test[0] = test[1] = test[2] = "X"   # X wins top row
print("Board with X winning top row:")
print_board(test)
print("Winner:", check_winner(test))
print("Empty cells:", get_empty_cells(test))
print("Board full?", is_board_full(test))

Board with X winning top row:

 X | X | X 
---+---+---
   |   |   
---+---+---
   |   |   

Winner: [0]
Empty cells: [3, 4, 5, 6, 7, 8]
Board full? False


In [16]:
import math

In [17]:
class MinimaxAI:


    def __init__(self,ai_mark=PLAYER_O):
        self.ai_mark = ai_mark
        self.human_mark= opponent(ai_mark)
        self.nodes_evaluated= 0
        self.nodes_pruned=0

    def best_moves(self,board):
        self.nodes_evaluated=0
        self.nodes_pruned=0

        best_score= -math.inf
        best_index= -1

        for idx in get_empty_cells(board):
            board[idx] = self.ai_mark                        
            score = self._minimax(board, 0, False,           
                                  -math.inf, math.inf)
            board[idx] = EMPTY

            if score>best_score:
                best_score=score
                best_index=idx

        return best_index


    def _minimax(self, board, depth, is_ai_turn, alpha, beta):
       
        winner = check_winner(board)
        if winner == self.ai_mark:    return 10 - depth  
        if winner == self.human_mark: return depth - 10  
        if is_board_full(board):      return 0

        if is_ai_turn:
            best = -math.inf
            for idx in get_empty_cells(board):
                board[idx] = self.ai_mark
                score = self._minimax(board, depth+1, False, alpha, beta)
                board[idx] = EMPTY
                best  = max(best, score)
                alpha = max(alpha, best)
                if beta <= alpha:
                    self.nodes_pruned += 1
                    break          
            return best

        else:
            best = math.inf
            for idx in get_empty_cells(board):
                board[idx] = self.human_mark
                score = self._minimax(board, depth+1, True, alpha, beta)
                board[idx] = EMPTY
                best = min(best, score)
                beta = min(beta, best)
                if beta <= alpha:
                    self.nodes_pruned += 1
                    break
            return best


        

In [18]:
print("Testing AI on a board where it can win immediately...\n")
test_board = ["X", "O", "X",
              "O", "O", " ",
              "X", " ", " "]
print_board(test_board)
ai = MinimaxAI(ai_mark=PLAYER_O)
move = ai.best_moves(test_board)
print(f"AI plays cell: {move}  ← should be 5 (winning move!)")
print(f"Nodes checked: {ai.nodes_evaluated}")
print(f"Branches skipped by pruning: {ai.nodes_pruned}")

Testing AI on a board where it can win immediately...


 X | O | X 
---+---+---
 O | O |   
---+---+---
 X |   |   

AI plays cell: 5  ← should be 5 (winning move!)
Nodes checked: 0
Branches skipped by pruning: 3


In [19]:
def get_human_move(board):
   
    while True:
        try:
            
            move = int(input("Your move (enter 0-8): "))

            if move not in range(9):
                print("  ⚠  Please enter a number between 0 and 8.")
            elif board[move] != EMPTY:
                print(f"  ⚠  Cell {move} is already taken! Try another.")
            else:
                return move   
                
        except ValueError:
            
            print("  ⚠  That's not a number. Try again.")

def show_position_guide():
    
    print("\nCell positions (type this number to play there):")
    print(" 0 | 1 | 2 ")
    print("---+---+---")
    print(" 3 | 4 | 5 ")
    print("---+---+---")
    print(" 6 | 7 | 8 ")
    print()


In [20]:
print("Position guide:")
show_position_guide()
print("(The input test is in the full game — Cell 5)")

Position guide:

Cell positions (type this number to play there):
 0 | 1 | 2 
---+---+---
 3 | 4 | 5 
---+---+---
 6 | 7 | 8 

(The input test is in the full game — Cell 5)


In [ ]:
def play_one_game(ai, human_mark):
    board        = make_board()
    ai_mark      = opponent(human_mark)
    current      = PLAYER_X       
    ai.ai_mark   = ai_mark
    ai.human_mark= human_mark

    show_position_guide()
    print_board(board)

    while True:
        
        if current == human_mark:
            print(f"Your turn ({human_mark})")
            move = get_human_move(board)
            board[move] = human_mark

        
        else:
            print(f"AI is thinking...")
            move = ai.best_move(board)
            board[move] = ai_mark
            print(f"AI played cell {move}")
            print(f"  → checked {ai.nodes_evaluated:,} moves, "
                  f"skipped {ai.nodes_pruned:,} branches")

        print_board(board)


        winner = check_winner(board)
        if winner:
            if winner == human_mark:
                return "human"
            else:
                return "ai"
        if is_board_full(board):
            return "draw"

        
        current = opponent(current)


def main():
    print("=" * 35)
    print("   TIC-TAC-TOE  —  Minimax AI")
    print("=" * 35)

    
    scores = {"human": 0, "ai": 0, "draw": 0}
    ai     = MinimaxAI()

    while True:
        
        while True:
            choice = input("Play as X (first) or O (second)? Enter X or O: ").upper()
            if choice in ("X", "O"):
                human_mark = choice
                break
            print("  ⚠  Enter X or O only.")

        
        result = play_one_game(ai, human_mark)

        
        scores[result] += 1

        if result == "human":
            print("🎉  You win! (The AI slipped up?)")
        elif result == "ai":
            print("🤖  AI wins! Minimax is unbeatable.")
        else:
            print("🤝  It's a draw!")

        
        print(f"\nScore → You: {scores['human']}  |  "
              f"AI: {scores['ai']}  |  Draws: {scores['draw']}")

        
        again = input("\nPlay again? (y / n): ").strip().lower()
        if again != "y":
            print("\nThanks for playing! Keep learning Python 🐍")
            break


main()

   TIC-TAC-TOE  —  Minimax AI


In [23]:
import math


EMPTY    = " "
PLAYER_X = "X"
PLAYER_O = "O"

WINNING_COMBOS = [
    (0,1,2),(3,4,5),(6,7,8),
    (0,3,6),(1,4,7),(2,5,8),
    (0,4,8),(2,4,6)
]


def make_board():
    return [" "] * 9

def print_board(board):
    print()
    print(f" {board[0]} | {board[1]} | {board[2]} ")
    print("---+---+---")
    print(f" {board[3]} | {board[4]} | {board[5]} ")
    print("---+---+---")
    print(f" {board[6]} | {board[7]} | {board[8]} ")
    print()

def check_winner(board):
    for a, b, c in WINNING_COMBOS:
        if board[a] != EMPTY and board[a] == board[b] == board[c]:
            return board[a]
    return None

def is_board_full(board):
    return EMPTY not in board

def get_empty_cells(board):
    return [i for i in range(9) if board[i] == EMPTY]

def opponent(player):
    return PLAYER_O if player == PLAYER_X else PLAYER_X

def show_position_guide():
    print("\nCell positions:")
    print(" 0 | 1 | 2 ")
    print("---+---+---")
    print(" 3 | 4 | 5 ")
    print("---+---+---")
    print(" 6 | 7 | 8 ")
    print()

# ── AI CLASS ───────────────────────────────────────────────────────────────
class MinimaxAI:

    def __init__(self, ai_mark=PLAYER_O):
        self.ai_mark         = ai_mark
        self.human_mark      = opponent(ai_mark)
        self.nodes_evaluated = 0
        self.nodes_pruned    = 0

    def best_move(self, board):
        self.nodes_evaluated = 0
        self.nodes_pruned    = 0
        best_score = -math.inf
        best_index = -1

        for idx in get_empty_cells(board):
            board[idx] = self.ai_mark
            score = self._minimax(board, 0, False, -math.inf, math.inf)
            board[idx] = EMPTY
            if score > best_score:
                best_score = score
                best_index = idx
        return best_index

    def _minimax(self, board, depth, is_ai_turn, alpha, beta):
        winner = check_winner(board)
        if winner == self.ai_mark:    return 10 - depth
        if winner == self.human_mark: return depth - 10
        if is_board_full(board):      return 0

        if is_ai_turn:
            best = -math.inf
            for idx in get_empty_cells(board):
                board[idx] = self.ai_mark
                score = self._minimax(board, depth+1, False, alpha, beta)
                board[idx] = EMPTY
                best  = max(best, score)
                alpha = max(alpha, best)
                if beta <= alpha:
                    self.nodes_pruned += 1
                    break
            return best
        else:
            best = math.inf
            for idx in get_empty_cells(board):
                board[idx] = self.human_mark
                score = self._minimax(board, depth+1, True, alpha, beta)
                board[idx] = EMPTY
                best = min(best, score)
                beta = min(beta, best)
                if beta <= alpha:
                    self.nodes_pruned += 1
                    break
            return best

# ── HUMAN INPUT ────────────────────────────────────────────────────────────
def get_human_move(board):
    while True:
        try:
            move = int(input("Your move (enter 0-8): "))
            if move not in range(9):
                print("  ⚠  Enter a number 0 to 8.")
            elif board[move] != EMPTY:
                print(f"  ⚠  Cell {move} is taken! Try another.")
            else:
                return move
        except ValueError:
            print("  ⚠  That's not a number. Try again.")

# ── GAME LOOP ──────────────────────────────────────────────────────────────
def play_one_game(ai, human_mark):
    board         = make_board()
    ai_mark       = opponent(human_mark)
    ai.ai_mark    = ai_mark
    ai.human_mark = human_mark
    current       = PLAYER_X

    show_position_guide()
    print_board(board)

    while True:
        if current == human_mark:
            print(f"Your turn ({human_mark})")
            move = get_human_move(board)
            board[move] = human_mark
        else:
            print("AI is thinking...")
            move = ai.best_move(board)
            board[move] = ai_mark
            print(f"AI played cell {move}  "
                  f"(checked {ai.nodes_evaluated:,} moves, "
                  f"skipped {ai.nodes_pruned:,} branches)")

        print_board(board)

        winner = check_winner(board)
        if winner:
            return "human" if winner == human_mark else "ai"
        if is_board_full(board):
            return "draw"

        current = opponent(current)

# ── MAIN ───────────────────────────────────────────────────────────────────
def main():
    print("=" * 35)
    print("   TIC-TAC-TOE  —  Minimax AI")
    print("=" * 35)

    scores = {"human": 0, "ai": 0, "draw": 0}
    ai     = MinimaxAI()

    while True:
        while True:
            choice = input("Play as X (first) or O (second)? ").upper()
            if choice in ("X", "O"):
                human_mark = choice
                break
            print("  ⚠  Enter X or O only.")

        result = play_one_game(ai, human_mark)
        scores[result] += 1

        if result == "human":
            print("🎉  You win!")
        elif result == "ai":
            print("🤖  AI wins! Minimax is unbeatable.")
        else:
            print("🤝  Draw!")

        print(f"\nScore → You: {scores['human']}  |  "
              f"AI: {scores['ai']}  |  Draws: {scores['draw']}")

        again = input("\nPlay again? (y / n): ").strip().lower()
        if again != "y":
            print("\nThanks for playing! Keep learning Python 🐍")
            break

# ▶ START
main()




   TIC-TAC-TOE  —  Minimax AI


Play as X (first) or O (second)?  X



Cell positions:
 0 | 1 | 2 
---+---+---
 3 | 4 | 5 
---+---+---
 6 | 7 | 8 


   |   |   
---+---+---
   |   |   
---+---+---
   |   |   

Your turn (X)


Your move (enter 0-8):  1



   | X |   
---+---+---
   |   |   
---+---+---
   |   |   

AI is thinking...
AI played cell 0  (checked 0 moves, skipped 2,585 branches)

 O | X |   
---+---+---
   |   |   
---+---+---
   |   |   

Your turn (X)


...


 O | X | X 
---+---+---
 O |   |   
---+---+---
 X |   |   

AI is thinking...
AI played cell 4  (checked 0 moves, skipped 8 branches)

 O | X | X 
---+---+---
 O | O |   
---+---+---
 X |   |   

Your turn (X)


Your move (enter 0-8):  5



 O | X | X 
---+---+---
 O | O | X 
---+---+---
 X |   |   

AI is thinking...
AI played cell 8  (checked 0 moves, skipped 0 branches)

 O | X | X 
---+---+---
 O | O | X 
---+---+---
 X |   | O 

🤖  AI wins! Minimax is unbeatable.

Score → You: 0  |  AI: 1  |  Draws: 0



Play again? (y / n):  n



Thanks for playing! Keep learning Python 🐍
